# FDPIS — Notebook 1: Data Preparation

**Run this first.** Produces `data/processed.parquet` for Notebook 2.

### Setup
Put every BTS monthly CSV inside `data/raw/`. Filenames don't matter — the loader globs the folder.
One month works; more is better. The code is identical either way.

### What this notebook does
1. Loads and merges all monthly CSVs
2. Rebuilds `FL_DATE` from integer Y/M/D (never trusts the date string)
3. Separates cancelled flights
4. Reconstructs each aircraft's daily rotation from `TAIL_NUM`
5. Derives a reactionary-delay flag and validates it against BTS ground truth
6. Saves to parquet

### Lessons baked in from the previous attempt
- Excel silently corrupts BTS dates → always rebuild from integers, never open the CSV in Excel
- A tail number can tangle two physically different aircraft → continuity check is mandatory
- BTS cause codes only exist for arrivals 15+ min late → validation must be restricted to those rows

## 1 — Imports and configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import os

pd.set_option('display.max_columns', 60)

RAW_DIR = 'data/raw'
OUT_PATH = 'data/processed.parquet'
CANCELLED_PATH = 'data/cancelled.parquet'

# Minimum physical turnaround by carrier, in minutes. Sourced from published
# airline turnaround figures; these are estimates, not ground truth — say so in the report.
MIN_TURNAROUND = {
    'WN': 30,
    'NK': 35, 'G4': 35, 'F9': 35,
    'AA': 40, 'DL': 40, 'UA': 40, 'B6': 40,
    'AS': 35,
    'OO': 25, 'YX': 25, 'MQ': 25, 'OH': 25,
}
DEFAULT_TURNAROUND = 40

os.makedirs(RAW_DIR, exist_ok=True)
print('Looking for CSVs in:', os.path.abspath(RAW_DIR))

Looking for CSVs in: D:\FDPIS\data\raw


## 2 — Load and merge every monthly CSV

Add more months by dropping more files into `data/raw/`. Nothing below needs to change.

In [2]:
files = sorted(glob(f'{RAW_DIR}/*.csv'))
if not files:
    raise FileNotFoundError(f'No CSVs in {RAW_DIR}/ — move your BTS files there.')

print(f'Found {len(files)} file(s):')
frames = []
for f in files:
    part = pd.read_csv(f)
    print(f'  {os.path.basename(f):<45} {len(part):>8,} rows')
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
del frames
print(f'\nMerged: {df.shape[0]:,} rows x {df.shape[1]} columns')

Found 3 file(s):


  2025 APRIL.csv                                 583,950 rows


  2025 JULY.csv                                  631,428 rows


  2025 Jan.csv                                   539,747 rows



Merged: 1,755,125 rows x 38 columns


## 3 — Verify the schema

Fail loudly now rather than silently three cells later.

In [3]:
REQUIRED = [
    'YEAR','MONTH','DAY_OF_MONTH','DAY_OF_WEEK','FL_DATE',
    'OP_UNIQUE_CARRIER','TAIL_NUM','OP_CARRIER_FL_NUM',
    'ORIGIN','ORIGIN_CITY_NAME','DEST','DEST_CITY_NAME',
    'CRS_DEP_TIME','DEP_TIME','DEP_DELAY','DEP_DELAY_NEW','DEP_DEL15',
    'TAXI_OUT','WHEELS_OFF','WHEELS_ON','TAXI_IN',
    'CRS_ARR_TIME','ARR_TIME','ARR_DELAY','ARR_DELAY_NEW','ARR_DEL15',
    'CANCELLED','CANCELLATION_CODE','DIVERTED',
    'CRS_ELAPSED_TIME','ACTUAL_ELAPSED_TIME','AIR_TIME','DISTANCE',
    'CARRIER_DELAY','WEATHER_DELAY','NAS_DELAY','SECURITY_DELAY','LATE_AIRCRAFT_DELAY',
]
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
print('Schema OK — all 38 required columns present.')

Schema OK — all 38 required columns present.


## 4 — Rebuild FL_DATE from integers

The raw `FL_DATE` string is unreliable: opening a BTS CSV in Excel silently reinterprets
day-of-month values 1–12 as month numbers (Jan 5 becomes May 1). `YEAR`, `MONTH` and
`DAY_OF_MONTH` are plain integers and cannot be corrupted this way.

In [4]:
df['FL_DATE'] = pd.to_datetime(dict(year=df['YEAR'], month=df['MONTH'], day=df['DAY_OF_MONTH']))

print('Unique dates :', df['FL_DATE'].nunique())
print('Range        :', df['FL_DATE'].min().date(), 'to', df['FL_DATE'].max().date())
print()
print('Rows per month:')
print(df.groupby([df['FL_DATE'].dt.year, df['FL_DATE'].dt.month]).size().to_string())

Unique dates : 92
Range        : 2025-01-01 to 2025-07-31

Rows per month:


FL_DATE  FL_DATE
2025     1          539747
         4          583950
         7          631428


## 5 — Data quality check

In [5]:
nulls = df.isna().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print('Columns with nulls:')
print((nulls.to_frame('nulls').assign(pct=lambda x: (100*x['nulls']/len(df)).round(2))).to_string())

print()
print('Carriers present:', df['OP_UNIQUE_CARRIER'].nunique())
print(df['OP_UNIQUE_CARRIER'].value_counts().to_string())

print()
print('Note: the five cause-code columns are only populated when arrival delay >= 15 min.')
print('Their high null rate is expected, not a data problem.')

Columns with nulls:
                       nulls    pct
CANCELLATION_CODE    1718426  97.91
NAS_DELAY            1366379  77.85
LATE_AIRCRAFT_DELAY  1366379  77.85
WEATHER_DELAY        1366379  77.85
CARRIER_DELAY        1366379  77.85
SECURITY_DELAY       1366379  77.85
AIR_TIME               42315   2.41
ACTUAL_ELAPSED_TIME    42315   2.41
ARR_DEL15              42315   2.41
ARR_DELAY_NEW          42315   2.41
ARR_DELAY              42315   2.41
ARR_TIME               37266   2.12
TAXI_IN                37266   2.12
WHEELS_ON              37266   2.12
WHEELS_OFF             36481   2.08
TAXI_OUT               36481   2.08
DEP_DELAY              35139   2.00
DEP_DELAY_NEW          35139   2.00
DEP_DEL15              35139   2.00
DEP_TIME               35021   2.00
TAIL_NUM                4728   0.27

Carriers present: 14
OP_UNIQUE_CARRIER
WN    348606
DL    254560
AA    245201
OO    210232
UA    196051
YX     86613
MQ     73722
OH     63619
AS     60807
B6     58610
NK     53154
F9   

## 6 — Separate cancelled flights

Cancelled flights have no actual times, so they'd corrupt rotation reconstruction.
They're saved separately — a cancellation is itself a cascade event worth modelling later.

In [6]:
cancelled = df[df['CANCELLED'] == 1].copy()
flown = df[df['CANCELLED'] == 0].copy()

print(f'Cancelled : {len(cancelled):>8,} ({100*len(cancelled)/len(df):.2f}%)')
print(f'Flown     : {len(flown):>8,}')

before = len(flown)
flown = flown.dropna(subset=['TAIL_NUM']).copy()
print(f'\nDropped {before - len(flown):,} flown rows with no TAIL_NUM (rotation impossible without it)')
print(f'Remaining : {len(flown):,}')

Cancelled :   36,699 (2.09%)
Flown     : 1,718,426



Dropped 0 flown rows with no TAIL_NUM (rotation impossible without it)
Remaining : 1,718,426


## 7 — Reconstruct aircraft rotations

Sorting by tail number, date and scheduled departure recovers the sequence of legs each
physical aircraft flew that day. `LEG_NUM` is its position in that sequence.

This is Layer 2: the aircraft-continuity edges of the dependency graph.

In [7]:
def hhmm_to_min(x):
    """BTS stores clock times as integers: 1358 -> 13*60+58 = 838 minutes past midnight."""
    if pd.isna(x):
        return np.nan
    x = int(x)
    return (x // 100) * 60 + (x % 100)

flown['CRS_DEP_MIN'] = flown['CRS_DEP_TIME'].apply(hhmm_to_min)
flown['CRS_ARR_MIN'] = flown['CRS_ARR_TIME'].apply(hhmm_to_min)
flown['ARR_TIME_MIN'] = flown['ARR_TIME'].apply(hhmm_to_min)

flown = flown.sort_values(['TAIL_NUM','FL_DATE','CRS_DEP_MIN']).reset_index(drop=True)

grp = flown.groupby(['TAIL_NUM','FL_DATE'])
flown['LEG_NUM'] = grp.cumcount() + 1
flown['TAIL_LEGS_TODAY'] = grp['LEG_NUM'].transform('max')
flown['LEG_FRACTION'] = flown['LEG_NUM'] / flown['TAIL_LEGS_TODAY'].clip(lower=1)
flown['IS_FIRST_LEG'] = (flown['LEG_NUM'] == 1).astype(int)
flown['IS_LAST_LEG'] = (flown['LEG_NUM'] == flown['TAIL_LEGS_TODAY']).astype(int)

first_dep = grp['CRS_DEP_MIN'].transform('min')
flown['MINUTES_INTO_TAIL_DAY'] = flown['CRS_DEP_MIN'] - first_dep

print('Legs per aircraft-day:')
print(flown['TAIL_LEGS_TODAY'].describe().to_string())

Legs per aircraft-day:
count    1.718426e+06
mean     4.699182e+00
std      1.649881e+00
min      1.000000e+00
25%      4.000000e+00
50%      5.000000e+00
75%      6.000000e+00
max      1.300000e+01


## 8 — Rotation continuity check

A tail number does not always identify one physical aircraft. In the previous run, `N486AS`
showed legs departing Portland and Honolulu six minutes apart — two different aircraft
sharing a registration string in the data.

The guard is simple: a rotation link is only real if the previous leg's destination equals
this leg's origin. Everything downstream depends on this holding.

In [8]:
flown['PREV_DEST'] = grp['DEST'].shift(1)
flown['PREV_ARR_DELAY'] = grp['ARR_DELAY'].shift(1)
flown['PREV_CRS_ARR_MIN'] = grp['CRS_ARR_MIN'].shift(1)
flown['PREV_ARR_TIME_MIN'] = grp['ARR_TIME_MIN'].shift(1)

flown['HAS_PREV_LEG'] = flown['PREV_DEST'].notna()
flown['CONTINUOUS_ROTATION'] = (flown['PREV_DEST'] == flown['ORIGIN'])

n_prev = flown['HAS_PREV_LEG'].sum()
n_broken = (flown['HAS_PREV_LEG'] & ~flown['CONTINUOUS_ROTATION']).sum()
print(f'Legs with a previous leg : {n_prev:,}')
print(f'Broken continuity        : {n_broken:,} ({100*n_broken/max(n_prev,1):.2f}%)')
print('\nThese are excluded from propagation — they are not genuine aircraft links.')

print('\nWorst offenders (tails with most broken links):')
broken = flown[flown['HAS_PREV_LEG'] & ~flown['CONTINUOUS_ROTATION']]
print(broken['TAIL_NUM'].value_counts().head(10).to_string())

Legs with a previous leg : 1,282,904
Broken continuity        : 15,260 (1.19%)

These are excluded from propagation — they are not genuine aircraft links.

Worst offenders (tails with most broken links):
TAIL_NUM
N957SW    25
N947SW    24
N679CA    23
N816SK    23
N986SW    22
N718EV    20
N930EV    20
N829SK    19
N832SK    19
N603SK    18


## 9 — Buffer and absorbable slack

`SCHEDULED_BUFFER_MIN` is the gap the airline itself planned between one leg's scheduled
arrival and the next leg's scheduled departure. Not all of it is absorbable — most is consumed
by the physical turnaround.

```
available_slack = scheduled_gap  −  minimum_turnaround
```

The previous version treated the whole scheduled gap as slack, which under-detected cascades
badly (recall 38.9%). Subtracting the physical turnaround is what fixed it.

In [9]:
# A flight crosses midnight when its scheduled arrival clock time is earlier
# than its scheduled departure clock time. Add a day to that leg's arrival.
flown['CRS_ARR_MIN_ADJ'] = flown['CRS_ARR_MIN'] + 1440 * (
    flown['CRS_ARR_MIN'] < flown['CRS_DEP_MIN']).astype(int)

grp2 = flown.groupby(['TAIL_NUM','FL_DATE'])
flown['PREV_CRS_ARR_MIN_ADJ'] = grp2['CRS_ARR_MIN_ADJ'].shift(1)

flown['SCHEDULED_BUFFER_MIN'] = flown['CRS_DEP_MIN'] - flown['PREV_CRS_ARR_MIN_ADJ']

# Negative now means a genuinely impossible rotation (overlapping legs, tail collision).
# Do NOT add 1440 — mark it broken and exclude it from propagation.
impossible = flown['SCHEDULED_BUFFER_MIN'] < 0
print(f'Impossible rotations (overlapping legs): {impossible.sum():,}')
flown.loc[impossible, 'CONTINUOUS_ROTATION'] = False
flown.loc[impossible, 'SCHEDULED_BUFFER_MIN'] = np.nan

# Beyond a few hours the aircraft is simply idle — extra slack changes nothing
# operationally, so cap it rather than letting the model key on extreme values.
SLACK_CAP = 240
flown['MIN_TURNAROUND'] = flown['OP_UNIQUE_CARRIER'].map(MIN_TURNAROUND).fillna(DEFAULT_TURNAROUND)
flown['AVAILABLE_SLACK'] = (
    flown['SCHEDULED_BUFFER_MIN'] - flown['MIN_TURNAROUND']).clip(lower=0, upper=SLACK_CAP)
flown['IS_TIGHT_TURNAROUND'] = (flown['AVAILABLE_SLACK'] < 15).astype(int)
flown['LONG_GROUND_TIME'] = (flown['SCHEDULED_BUFFER_MIN'] > SLACK_CAP).astype(int)

print()
print('Scheduled buffer (min):')
print(flown['SCHEDULED_BUFFER_MIN'].describe().to_string())
print()
print('Available slack (min, capped at 240):')
print(flown['AVAILABLE_SLACK'].describe().to_string())

Impossible rotations (overlapping legs): 23,560



Scheduled buffer (min):
count    1.259344e+06
mean     7.824068e+01
std      9.325323e+01
min      0.000000e+00
25%      4.500000e+01
50%      5.600000e+01
75%      7.100000e+01
max      1.151000e+03

Available slack (min, capped at 240):


count    1.259344e+06
mean     3.683215e+01
std      4.864174e+01
min      0.000000e+00
25%      1.500000e+01
50%      2.200000e+01
75%      3.600000e+01
max      2.400000e+02


## 10 — Derive the reactionary-delay flag

A delay is reactionary when the inbound aircraft's lateness exceeded the absorbable slack,
on a genuinely continuous rotation.

US data has `LATE_AIRCRAFT_DELAY` to check this against. Indian data will not — which is
precisely why the rule is validated here, where an answer key exists.

In [10]:
flown['IS_REACTIONARY'] = (
    flown['CONTINUOUS_ROTATION'] &
    (flown['PREV_ARR_DELAY'] > flown['AVAILABLE_SLACK'])
).astype(int)

# Predicted propagated minutes: what spills past the slack
flown['PREDICTED_PROP_MIN'] = np.where(
    flown['CONTINUOUS_ROTATION'],
    (flown['PREV_ARR_DELAY'] - flown['AVAILABLE_SLACK']).clip(lower=0),
    0.0
)

delayed = (flown['DEP_DELAY'] > 0).sum()
print(f'Reactionary flagged      : {flown["IS_REACTIONARY"].sum():,}')
print(f'Total delayed departures : {delayed:,}')
print(f'Reactionary share        : {100*flown["IS_REACTIONARY"].sum()/max(delayed,1):.1f}%')

Reactionary flagged      : 188,176
Total delayed departures : 656,287
Reactionary share        : 28.7%


## 11 — Validate against BTS ground truth

**Restrict to rows where cause codes exist.** BTS only records them for arrivals 15+ minutes
late, so scoring against all rows treats every sub-threshold cascade as a false positive.
In the previous run this alone made the difference between 64.8% and 86.7% precision —
the unrestricted number was measuring BTS's reporting floor, not our logic.

In [11]:
has_cause = flown['LATE_AIRCRAFT_DELAY'].notna()
sub = flown[has_cause]
truth = (sub['LATE_AIRCRAFT_DELAY'] > 0).astype(int)
pred = sub['IS_REACTIONARY']

tp = int(((pred==1)&(truth==1)).sum()); fp = int(((pred==1)&(truth==0)).sum())
fn = int(((pred==0)&(truth==1)).sum()); tn = int(((pred==0)&(truth==0)).sum())
prec = tp/max(tp+fp,1); rec = tp/max(tp+fn,1)
f1 = 2*prec*rec/max(prec+rec,1e-9)

print(f'Rows with cause codes: {has_cause.sum():,} of {len(flown):,} '
      f'({100*has_cause.mean():.1f}%)')
print()
print(f'Precision : {prec:.3f}')
print(f'Recall    : {rec:.3f}')
print(f'F1        : {f1:.3f}')
print()
print(pd.DataFrame([[tn,fp],[fn,tp]],
                   index=['Ours: not reactionary','Ours: reactionary'],
                   columns=['BTS: not reactionary','BTS: reactionary']).to_string())

# Magnitude accuracy, measured only on real cascades — averaging over the
# mostly-zero majority flatters the number and hides the real error.
cascades = sub[sub['LATE_AIRCRAFT_DELAY'] > 0]
mae = (cascades['PREDICTED_PROP_MIN'] - cascades['LATE_AIRCRAFT_DELAY']).abs().mean()
corr = cascades['PREDICTED_PROP_MIN'].corr(cascades['LATE_AIRCRAFT_DELAY'])
print()
print(f'Propagation MAE on {len(cascades):,} real cascades: {mae:.1f} min')
print(f'Correlation: {corr:.3f}')
print('\nThis is the Layer 4 arithmetic baseline the learned model must beat.')

Rows with cause codes: 388,746 of 1,718,426 (22.6%)

Precision : 0.894
Recall    : 0.668
F1        : 0.765

                       BTS: not reactionary  BTS: reactionary
Ours: not reactionary                177608             15563
Ours: reactionary                     64881            130694



Propagation MAE on 195,575 real cascades: 26.0 min
Correlation: 0.518

This is the Layer 4 arithmetic baseline the learned model must beat.


## 12 — Schedule-derived congestion features

Counts of *scheduled* flights only — no outcome information involved, so computing them
across the whole dataset is safe and doesn't leak.

In [12]:
flown['DEP_HOUR'] = (flown['CRS_DEP_TIME'] // 100).astype(int)
flown['ARR_HOUR'] = (flown['CRS_ARR_TIME'] // 100).astype(int)
flown['IS_WEEKEND'] = flown['DAY_OF_WEEK'].isin([6,7]).astype(int)

flown['ORIGIN_HOUR_DEPARTURES'] = flown.groupby(
    ['ORIGIN','FL_DATE','DEP_HOUR'])['ORIGIN'].transform('size')
flown['DEST_HOUR_ARRIVALS'] = flown.groupby(
    ['DEST','FL_DATE','ARR_HOUR'])['DEST'].transform('size')
flown['ORIGIN_DAY_DEPARTURES'] = flown.groupby(
    ['ORIGIN','FL_DATE'])['ORIGIN'].transform('size')
flown['CARRIER_ORIGIN_FLIGHTS'] = flown.groupby(
    ['OP_UNIQUE_CARRIER','ORIGIN','FL_DATE'])['ORIGIN'].transform('size')
flown['CARRIER_ORIGIN_SHARE'] = (
    flown['CARRIER_ORIGIN_FLIGHTS'] / flown['ORIGIN_DAY_DEPARTURES'].clip(lower=1))

flown['SPEED_PROXY'] = flown['DISTANCE'] / flown['CRS_ELAPSED_TIME'].clip(lower=1)

print('Congestion features added.')
print(flown[['ORIGIN_HOUR_DEPARTURES','DEST_HOUR_ARRIVALS',
             'CARRIER_ORIGIN_SHARE']].describe().to_string())

Congestion features added.


       ORIGIN_HOUR_DEPARTURES  DEST_HOUR_ARRIVALS  CARRIER_ORIGIN_SHARE
count            1.718426e+06        1.718426e+06          1.718426e+06
mean             2.330372e+01        2.260504e+01          3.235377e-01
std              2.024554e+01        2.058175e+01          2.580074e-01
min              1.000000e+00        1.000000e+00          1.007049e-03
25%              7.000000e+00        6.000000e+00          1.194030e-01
50%              1.800000e+01        1.700000e+01          2.452830e-01
75%              3.300000e+01        3.100000e+01          4.978992e-01
max              1.020000e+02        1.030000e+02          1.000000e+00


## 13 — Sanity check: eyeball one aircraft's day

In [13]:
busy = flown[flown['TAIL_LEGS_TODAY'] >= 5]
if len(busy):
    t = busy.iloc[0]['TAIL_NUM']; d = busy.iloc[0]['FL_DATE']
    cols = ['LEG_NUM','ORIGIN','DEST','CRS_DEP_TIME','CRS_ARR_TIME',
            'DEP_DELAY','ARR_DELAY','SCHEDULED_BUFFER_MIN','AVAILABLE_SLACK',
            'CONTINUOUS_ROTATION','IS_REACTIONARY']
    print(f'Tail {t} on {d.date()}:\n')
    print(flown[(flown['TAIL_NUM']==t) & (flown['FL_DATE']==d)][cols].to_string(index=False))
    print('\nCheck: does each DEST match the next ORIGIN? Do the times run forward?')

Tail 188NV on 2025-01-02:

 LEG_NUM ORIGIN DEST  CRS_DEP_TIME  CRS_ARR_TIME  DEP_DELAY  ARR_DELAY  SCHEDULED_BUFFER_MIN  AVAILABLE_SLACK  CONTINUOUS_ROTATION  IS_REACTIONARY
       1    FLL  LEX           600           828       -3.0      -21.0                   NaN              NaN                False               0
       2    LEX  FLL           918          1145       -9.0      -21.0                  50.0             15.0                 True               0
       3    FLL  USA          1235          1435       -4.0        1.0                  50.0             15.0                 True               0
       4    USA  FLL          1525          1726       17.0       15.0                  50.0             15.0                 True               0
       5    FLL  TYS          1816          2020       28.0       31.0                  50.0             15.0                 True               0
       6    TYS  FLL          2110          2320       32.0       45.0                  50.

## 14 — Save

In [14]:
flown.to_parquet(OUT_PATH, index=False)
cancelled.to_parquet(CANCELLED_PATH, index=False)

print(f'Saved {len(flown):,} rows -> {OUT_PATH} '
      f'({os.path.getsize(OUT_PATH)/1e6:.1f} MB)')
print(f'Saved {len(cancelled):,} rows -> {CANCELLED_PATH}')
print(f'\nDate span: {flown["FL_DATE"].min().date()} to {flown["FL_DATE"].max().date()} '
      f'({flown["FL_DATE"].nunique()} days)')
print('\nNext: run Notebook 2 (fdpis_2_modelling.ipynb)')

Saved 1,718,426 rows -> data/processed.parquet (94.1 MB)
Saved 36,699 rows -> data/cancelled.parquet

Date span: 2025-01-01 to 2025-07-31 (92 days)

Next: run Notebook 2 (fdpis_2_modelling.ipynb)
